# Week 10: SAR Flood Detection & Sensor Fusion — ARIA v7.0
## 第十週：SAR 淹水偵測與多源融合 — ARIA v7.0（全天候決策引擎）

**Course**: 遙測與空間資訊之分析與應用 | Remote Sensing Analysis & Applications  
**Institution**: National Taiwan University (NTU)  
**Instructor**: Prof. Su Wen-Ray (蘇文瑞教授)  
**Case**: 2025 馬太鞍溪堰塞湖 — SAR 全天候監測  

---

### Lab Rhythm / 實驗時間分配

| Lab | Topic | Duration | 主題 |
|-----|-------|----------|------|
| **Lab 1** | SAR 淹水偵測（STAC → Sentinel-1） | 35 min | 災前/災後 VV 對比 |
| **Lab 2** | ARIA v7.0 Sensor Fusion | 30 min | SAR + Optical 融合 |

### W8–W9 → W10 延續

| Week | Sensor | STAC Collection | Key Difference |
|------|--------|-----------------|----------------|
| W8–W9 | Sentinel-2（光學）| `sentinel-2-l2a` | 受雲遮蔽 → 需 SCL 雲遮罩 |
| **W10** | **Sentinel-1（SAR）** | **`sentinel-1-rtc`** | **穿透雲層 → 不需要雲量過濾！** |

同樣的 `pystac_client` → `stackstac` → `xarray` pipeline，只是感測器不同！

### 堰塞湖事件時序

- 2025/7/18–19 薇帕颱風 → 上游崩塌形成堰塞湖
- 7/23 面積 4 ha → 8月擴大 → 9/21 達 80 ha → 9/23 潰決
- **本週任務：用 SAR 偵測潰決前堰塞湖的水體範圍**

## Lab 1: SAR Flood Detection — Piercing the Cloud
### 實驗 1：SAR 淹水偵測 — 穿透雲層

### Why Are dB Values Negative? / 為什麼 dB 是負數？

SAR backscatter 以 **σ⁰** (sigma-nought) 表示 — 回波能量與發射能量的比值。

**公式：** `dB = 10 × log₁₀(σ⁰)`

| σ⁰ (linear) | dB | 地表類型 |
|---|---|---|
| 1.0 | 0 dB | 完美反射（理論值）|
| 0.1 | **-10 dB** | 粗糙地表 |
| 0.01 | **-20 dB** | 平滑水面（鏡面反射）|
| 0.001 | **-30 dB** | 極平滑表面 |

### Why -18 dB? / 為什麼閾值是 -18 dB？

| Surface | Typical VV (dB) | Scattering | 中文 |
|---|---|---|---|
| Calm water | -25 to -20 | Specular reflection | 鏡面反射 |
| Rough water | -15 to -10 | Partial diffuse | 部分散射 |
| Bare soil | -10 to -5 | Surface scattering | 表面散射 |
| Forest | -8 to -3 | Volume scattering | 體散射 |
| Buildings | > 0 | Double-bounce | 雙次彈跳 |

**-18 dB 介於平靜水面（< -20）和粗糙地面（> -10）之間，能捕捉大部分積水。**

In [ ]:
# [S1] Environment + STAC Setup
# ── 與 W8 Cell [1] 相同模式
# ──────────────────────────────────────────────────────────────────
import os, time, warnings
warnings.filterwarnings('ignore')

# GDAL HTTP retry（同 W8）
os.environ.setdefault('GDAL_HTTP_MAX_RETRY', '5')
os.environ.setdefault('GDAL_HTTP_RETRY_DELAY', '2')
os.environ.setdefault('GDAL_HTTP_TIMEOUT', '60')
os.environ.setdefault('GDAL_HTTP_MULTIRANGE', 'YES')
os.environ.setdefault('GDAL_HTTP_MERGE_CONSECUTIVE_RANGES', 'YES')
os.environ.setdefault('VSI_CACHE', 'TRUE')
os.environ.setdefault('VSI_CACHE_SIZE', '1000000000')
os.environ.setdefault('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', '.tif,.TIF,.tiff')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import median_filter
import xarray as xr
import rioxarray as rxr
import pystac_client
import planetary_computer as pc
import stackstac

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'PingFang TC', 'Heiti TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=pc.sign_inplace,
)

# ── 堰塞湖 bbox（擴大範圍以涵蓋完整湖域 + 周邊地形）──
LAKE_BBOX = [121.270, 23.685, 121.310, 23.715]

# ── 軌道方向 ──
# 升軌 ascending：雷達從西往東看 | 降軌 descending：雷達從東往西看
# 災前災後必須相同方向，否則差異圖充滿幾何偽影！
# （[S2] 會自動偵測可用的軌道方向）
SAR_THRESHOLD = -14   # dB — 寬鬆門檻，搭配形態學後處理去除雜訊

OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_compute(lazy_arr, tries=4):
    """Retry wrapper for COG reads（同 W8）。"""
    from rasterio.errors import RasterioIOError
    last_err = None
    for attempt in range(tries):
        try:
            return lazy_arr.compute()
        except (RasterioIOError, RuntimeError) as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f'COG read failed after {tries} attempts: {last_err}')

print('✅ Environment ready')
print(f'   LAKE_BBOX: {LAKE_BBOX}')
print(f'   SAR_THRESHOLD: {SAR_THRESHOLD} dB')

In [ ]:
# [S2] Search Sentinel-1 RTC — 搜尋 SAR 影像
# ──────────────────────────────────────────────────────────────────
# W8 用 robust_search() 搜尋 sentinel-2-l2a，需要過濾雲量。
# W10 搜尋 sentinel-1-rtc，不需要雲量過濾 — 這就是 SAR 的威力！
# ⚠ 但必須過濾軌道方向！升降軌觀測幾何不同，不能混用。
#
# TODO: 填入正確的 collection 名稱
# ──────────────────────────────────────────────────────────────────

def search_sar(bbox, datetime_range, orbit_state=None, max_items=30, tries=3):
    """Search sentinel-1-rtc with retry + orbit filtering."""
    last_err = None
    for attempt in range(tries):
        try:
            search = catalog.search(
                collections=[___],        # <-- TODO: 填入 'sentinel-1-rtc'
                bbox=bbox,
                datetime=datetime_range,
                max_items=max_items,
            )
            items = list(search.items())
            if orbit_state:
                items = [i for i in items
                         if i.properties.get('sat:orbit_state', '').lower() == orbit_state.lower()]
            items.sort(key=lambda i: i.properties.get('datetime', ''))
            return items
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f'STAC search failed: {last_err}')

# ── Step 1: 搜尋全部場景，看可用軌道方向 ──
all_pre  = search_sar(LAKE_BBOX, '2025-06-15/2025-07-17')
all_post = search_sar(LAKE_BBOX, '2025-09-10/2025-09-25')

print(f'Pre-disaster: {len(all_pre)} scenes')
for it in all_pre:
    p = it.properties
    print(f"  {p.get('datetime','?')[:16]} | {p.get('sat:orbit_state','?')} | {it.id[:40]}")

print(f'\nPost-disaster: {len(all_post)} scenes')
for it in all_post:
    p = it.properties
    print(f"  {p.get('datetime','?')[:16]} | {p.get('sat:orbit_state','?')} | {it.id[:40]}")

# ── Step 2: 自動選定共同軌道方向 ──
orbits_pre  = set(i.properties.get('sat:orbit_state','').lower() for i in all_pre)
orbits_post = set(i.properties.get('sat:orbit_state','').lower() for i in all_post)
common = orbits_pre & orbits_post
print(f'\n共同軌道: {common}')

if 'ascending' in common:
    ORBIT_STATE = 'ascending'
elif 'descending' in common:
    ORBIT_STATE = 'descending'
elif common:
    ORBIT_STATE = list(common)[0]
else:
    ORBIT_STATE = list(orbits_post)[0] if orbits_post else None
    print(f'⚠ 沒有共同軌道！使用 {ORBIT_STATE}')

print(f'→ 選定: {ORBIT_STATE}')

items_pre  = [i for i in all_pre  if i.properties.get('sat:orbit_state','').lower() == ORBIT_STATE]
items_post = [i for i in all_post if i.properties.get('sat:orbit_state','').lower() == ORBIT_STATE]
print(f'PRE ({ORBIT_STATE}): {len(items_pre)} | POST ({ORBIT_STATE}): {len(items_post)}')

In [ ]:
# [S3] stream_sar — 串流讀取 SAR 資料
# ──────────────────────────────────────────────────────────────────
# 跟 W8 stream_cube() 的三個差異：
#   1. collection = sentinel-1-rtc（不是 sentinel-2-l2a）
#   2. assets = ['vv'] 或 ['vh']（不是 B02–B12）
#   3. 不除以 10000（SAR 已經是 calibrated backscatter）
#
# TODO: 填入正確的 assets 參數
# ──────────────────────────────────────────────────────────────────

def stream_sar(item, bands=['vv'], bbox=LAKE_BBOX):
    """Stream Sentinel-1 GRD as lazy xarray (linear backscatter)."""
    signed = pc.sign(item)
    cube = stackstac.stack(
        [signed],
        assets=___,          # <-- TODO: 填入 bands
        epsg=32651,          # UTM 51N（同 W8）
        resolution=10,
        bounds_latlon=bbox,
        chunksize=2048,
    ).squeeze('time')        # 注意：不需要 / 10000.0！
    return cube

print('✅ stream_sar() ready')

In [ ]:
# [S4] Load & Convert to dB
# ──────────────────────────────────────────────────────────────────
# SAR 的值是線性 backscatter，需要轉成 dB：
#   dB = 10 × log₁₀(linear_value)
#
# TODO: 載入災前/災後 VV，轉換為 dB
# ──────────────────────────────────────────────────────────────────

if not items_pre or not items_post:
    raise ValueError('災前或災後無場景！請回 [S2] 檢查')

pre_item = items_pre[0]
post_item = items_post[0]
print(f'PRE:  {pre_item.properties["datetime"][:16]} | {pre_item.properties.get("sat:orbit_state","?")}')
print(f'POST: {post_item.properties["datetime"][:16]} | {post_item.properties.get("sat:orbit_state","?")}')

# 確認軌道一致
orb_pre = pre_item.properties.get('sat:orbit_state', '')
orb_post = post_item.properties.get('sat:orbit_state', '')
if orb_pre.lower() != orb_post.lower():
    print(f'⚠ WARNING: orbit mismatch! {orb_pre} vs {orb_post}')
else:
    print(f'✅ Orbit match: both {orb_post}')

# Pre-disaster VV
print('\nLoading pre-disaster VV...')
vv_pre_linear = safe_compute(stream_sar(pre_item, ['vv']))
vv_pre_db = ___  # <-- TODO: 10 * np.log10(vv_pre_linear.values.squeeze().astype(np.float32))
vv_pre_db = np.where(np.isfinite(vv_pre_db), vv_pre_db, np.nan)

# Post-disaster VV
print('Loading post-disaster VV...')
vv_post_linear = safe_compute(stream_sar(post_item, ['vv']))
vv_post_db = ___  # <-- TODO: 10 * np.log10(vv_post_linear.values.squeeze().astype(np.float32))
vv_post_db = np.where(np.isfinite(vv_post_db), vv_post_db, np.nan)

print(f'\nPre  shape: {vv_pre_db.shape}, range: {np.nanmin(vv_pre_db):.1f} ~ {np.nanmax(vv_pre_db):.1f} dB')
print(f'Post shape: {vv_post_db.shape}, range: {np.nanmin(vv_post_db):.1f} ~ {np.nanmax(vv_post_db):.1f} dB')
print('\n✅ All SAR loaded via STAC — no files downloaded!')

In [ ]:
# [S5] Visualize Pre vs Post SAR
# ──────────────────────────────────────────────────────────────────
# TODO: 建立 1×3 子圖：災前 VV / 災後 VV / 差異圖
#   - cmap='gray', vmin=-30, vmax=0
#   - 差異 = post - pre，cmap='RdBu'
#   - 藍色 = backscatter 下降 = 新增水體
# ──────────────────────────────────────────────────────────────────

pre_date = pre_item.properties['datetime'][:10]
post_date = post_item.properties['datetime'][:10]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(___, cmap='gray', vmin=-30, vmax=0)  # <-- TODO: vv_pre_db
axes[0].set_title(f'Pre: {pre_date} VV (dB)')

axes[1].imshow(___, cmap='gray', vmin=-30, vmax=0)  # <-- TODO: vv_post_db
axes[1].set_title(f'Post: {post_date} VV (dB)')

diff_db = ___  # <-- TODO: vv_post_db - vv_pre_db
im = axes[2].imshow(diff_db, cmap='RdBu', vmin=-15, vmax=15)
axes[2].set_title('Difference (Post − Pre)')
plt.colorbar(im, ax=axes[2], shrink=0.8, label='ΔdB')

for ax in axes:
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.suptitle('堰塞湖 SAR 時序對比', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/W10_sar_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

### Speckle Filtering: Why It Matters / 斑點雜訊濾波

SAR 影像含有 **speckle noise** — 同調波干涉產生的「椒鹽」雜訊。

**Captain's Rule: NEVER threshold raw SAR!** 斑點雜訊會產生大量假水體像素。

```
Raw SAR → Median Filter (5×5) → Filtered SAR → Threshold → Water Mask
```

In [ ]:
# [S6] Speckle Filter + Threshold + Morphological Cleanup
# ── SAR 水體偵測三步驟：
#    Step 1: Median filter 去除 speckle 雜訊
#    Step 2: 寬鬆門檻 (VV < -14 dB) 抓取完整水體
#    Step 3: Morphological opening + connected component 去除假陽性

from scipy.ndimage import binary_opening, label

# Step 1: Speckle filter
sar_filtered = median_filter(vv_post_db, size=5)

# Step 2: 寬鬆門檻
sar_water_raw = (sar_filtered < SAR_THRESHOLD).astype(np.uint8)
raw_count = np.sum(sar_water_raw)

# Step 3a: Morphological opening（先侵蝕再膨脹，去除零碎雜訊）
struct = np.ones((3, 3))  # 3×3 結構元素
sar_water_opened = binary_opening(sar_water_raw, structure=struct, iterations=1).astype(np.uint8)
opened_count = np.sum(sar_water_opened)

# Step 3b: Connected component filtering（只保留大面積水體）
MIN_WATER_PIXELS = 50  # 50 px × 100 m² = 0.5 ha
labeled, n_features = label(sar_water_opened)
sar_water = np.zeros_like(sar_water_opened)
kept = 0
removed_regions = 0
for region_id in range(1, n_features + 1):
    region_size = np.sum(labeled == region_id)
    if region_size >= MIN_WATER_PIXELS:
        sar_water[labeled == region_id] = 1
        kept += 1
    else:
        removed_regions += 1

# Pixel area (10 m × 10 m = 100 m²)
px_area_ha = (10 * 10) / 10000  # hectares
water_n = np.sum(sar_water)
flood_ha = water_n * px_area_ha

print(f'Step 1: Median filter (5×5) — 去除 speckle')
print(f'Step 2: Threshold VV < {SAR_THRESHOLD} dB → {raw_count:,} px ({raw_count * px_area_ha:.1f} ha)')
print(f'Step 3a: Morphological opening → {opened_count:,} px （去除 {raw_count - opened_count:,} 零碎像素）')
print(f'Step 3b: Connected components → {n_features} regions found')
print(f'         Kept {kept} regions (≥ {MIN_WATER_PIXELS} px = {MIN_WATER_PIXELS * px_area_ha:.1f} ha)')
print(f'         Removed {removed_regions} small fragments')
print(f'\n✅ Final water mask: {water_n:,} px = {flood_ha:.1f} ha')
print(f'   (NCDR 同期實測約 80 ha)')
print(f'   Mean dB in flood zone: {np.nanmean(sar_filtered[sar_water==1]):.1f} dB')

In [ ]:
# [S7] Lab 1 Visualization: 2×2 Panel
# ──────────────────────────────────────────────────────────────────
# TODO: 建立 2×2 面板
#   (a) Raw SAR VV    (b) Filtered SAR
#   (c) Flood Mask     (d) SAR + Overlay
# ──────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0,0].imshow(vv_post_db, cmap='gray', vmin=-30, vmax=0)
axes[0,0].set_title(f'(a) Raw SAR VV — {post_date}')

axes[0,1].imshow(___, cmap='gray', vmin=-30, vmax=0)  # <-- TODO: sar_filtered
axes[0,1].set_title('(b) Filtered (Median 5×5)')

axes[1,0].imshow(___, cmap='Blues', vmin=0, vmax=1)    # <-- TODO: sar_water
axes[1,0].set_title(f'(c) Water Mask (VV < {SAR_THRESHOLD} dB)')

axes[1,1].imshow(sar_filtered, cmap='gray', vmin=-30, vmax=0)
axes[1,1].imshow(sar_water, cmap='Blues', alpha=0.4)
axes[1,1].set_title('(d) SAR + Water Overlay')

for ax in axes.flat:
    ax.set_xlabel('Column'); ax.set_ylabel('Row')

plt.suptitle(f'Lab 1: SAR Flood Detection — ARIA v7.0\n堰塞湖 VV < {SAR_THRESHOLD} dB',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/W10_L1_sar_flood.png', dpi=150, bbox_inches='tight')
plt.show()

### Discussion: What Did the Radar See That Optical Missed?
### 討論：雷達看到了什麼，光學漏掉了什麼？

**觀察重點：**

1. **Cloud penetration**: SAR 在颱風季仍偵測到堰塞湖，Sentinel-2 可能整張都是白雲
2. **Water surface**: VV < threshold 的區域 = 鏡面反射 = 水體
3. **Speckle effect**: 未濾波直接做閾值 → 產生大量假水體像素
4. **Area underestimation**: SAR 側視幾何在山谷會低估水體面積

**思考題：**
- 為什麼 -18 dB 是合理的閾值？用 -12 dB 或 -25 dB 會怎樣？
- 只用 SAR 不用光學，能不能偵測到堰塞湖？

---
## Lab 2: ARIA v7.0 — Multi-Source Sensor Fusion
### 實驗 2：ARIA v7.0 — 多源資料融合

**Objective**: 結合 SAR 水體遮罩 + Sentinel-2 光學 NDWI + 地形校正 → ARIA v7.0 信心圖

| Code | Label | Condition |
|------|-------|-----------|
| 3 | High Confidence | SAR ✓ + NDWI ✓ |
| 2 | SAR Only (Cloudy) | SAR ✓ + Cloud ✓ |
| 1 | Optical Only | NDWI ✓ + SAR ✗ |
| 0 | No Detection | Both ✗ |

In [ ]:
# [S8] Search Sentinel-2 for Optical Comparison
# ──────────────────────────────────────────────────────────────────
# 同時期搜尋 Sentinel-2 — 用 W8 的 robust_search 模式
# 如果全被雲遮蔽 → 正好說明 SAR 的必要性
# ──────────────────────────────────────────────────────────────────

def robust_search(bbox, datetime_range, cloud_max=80, max_items=20, tries=3):
    """Same as W8 — search sentinel-2-l2a with client-side cloud filter."""
    last_err = None
    for attempt in range(tries):
        try:
            search = catalog.search(
                collections=['sentinel-2-l2a'],
                bbox=bbox, datetime=datetime_range, max_items=max_items,
            )
            items = list(search.items())
            items = [i for i in items if i.properties.get('eo:cloud_cover', 100) < cloud_max]
            items.sort(key=lambda i: i.properties['eo:cloud_cover'])
            return items
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f'Search failed: {last_err}')

items_s2 = robust_search(LAKE_BBOX, '2025-09-10/2025-09-30', cloud_max=80)
print(f'Sentinel-2 scenes: {len(items_s2)}')
for it in items_s2:
    print(f"  {it.properties['datetime'][:10]} | cloud: {it.properties.get('eo:cloud_cover',-1):.0f}%")

In [ ]:
# [S9] Build Optical + Cloud Masks
# ──────────────────────────────────────────────────────────────────
# 如果 Sentinel-2 可用 → NDWI + SCL（同 W8/W9 模式）
# 如果不可用 → 100% 雲覆蓋（SAR 是唯一資料源）
# ──────────────────────────────────────────────────────────────────

H, W = vv_post_db.shape

if items_s2:
    s2_item = items_s2[0]
    print(f'Using: {s2_item.properties["datetime"][:10]}')
    
    def stream_cube(item, bands=['B03', 'B08'], bbox=LAKE_BBOX):
        signed = pc.sign(item)
        return stackstac.stack(
            [signed], assets=bands, epsg=32651, resolution=10,
            bounds_latlon=bbox, chunksize=2048,
        ).squeeze('time') / 10000.0
    
    cube_s2 = stream_cube(s2_item)
    green = safe_compute(cube_s2.sel(band='B03')).values
    nir = safe_compute(cube_s2.sel(band='B08')).values
    ndwi = (green - nir) / (green + nir + 1e-9)
    NDWI_THRESHOLD = 0.0  # 濁水降低閾值
    ndwi_mask = (ndwi > NDWI_THRESHOLD).astype(np.uint8)
    
    def stream_scl(item, bbox=LAKE_BBOX):
        signed = pc.sign(item)
        return safe_compute(stackstac.stack(
            [signed], assets=['SCL'], epsg=32651, resolution=10,
            bounds_latlon=bbox, chunksize=2048,
        ).squeeze('time').squeeze('band'))
    
    scl = stream_scl(s2_item)
    cloud_mask = (~np.isin(scl.values, [2,4,5,6,7,11])).astype(np.uint8)
    
    from scipy.ndimage import zoom
    if ndwi.shape != (H, W):
        ndwi_mask = zoom(ndwi_mask, (H/ndwi.shape[0], W/ndwi.shape[1]), order=0)
        cloud_mask = zoom(cloud_mask, (H/cloud_mask.shape[0], W/cloud_mask.shape[1]), order=0)
    
    print(f'NDWI water: {np.sum(ndwi_mask):,} px | Cloud: {np.mean(cloud_mask)*100:.0f}%')
else:
    cloud_mask = np.ones((H, W), dtype=np.uint8)
    ndwi_mask = np.zeros((H, W), dtype=np.uint8)
    print('⚠ No Sentinel-2 → 100% cloud (SAR is ONLY source)')

In [ ]:
# [S9b] 光學 vs SAR 對比圖
# ──────────────────────────────────────────────────────────────────
# 觀察：光學看不到什麼（被雲遮蔽），SAR 看到什麼（穿透雲層）
# ──────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

axes[0,0].imshow(vv_post_db, cmap='gray', vmin=-30, vmax=0)
axes[0,0].imshow(sar_water, cmap='Blues', alpha=0.4)
axes[0,0].set_title(f'(a) SAR Water — 不受雲影響')

axes[0,1].imshow(cloud_mask, cmap='Oranges', vmin=0, vmax=1)
axes[0,1].set_title(f'(b) Cloud Mask — {np.mean(cloud_mask)*100:.0f}% 雲覆蓋')

if items_s2:
    axes[1,0].imshow(ndwi, cmap='RdYlBu', vmin=-0.5, vmax=0.5)
    axes[1,0].set_title(f'(c) NDWI — 水體: {np.sum(ndwi_mask):,} px')
    
    overlay = np.zeros((*sar_water.shape, 3))
    overlay[cloud_mask == 1] = [1.0, 0.8, 0.5]
    overlay[ndwi_mask == 1]  = [0.2, 0.8, 0.2]
    overlay[sar_water == 1]  = [0.2, 0.4, 1.0]
    overlay[(ndwi_mask == 1) & (sar_water == 1)] = [0.8, 0.1, 0.1]
    axes[1,1].imshow(overlay)
    axes[1,1].set_title('(d) 紅=雙重確認 藍=SAR 綠=光學 橘=雲')
else:
    axes[1,0].text(0.5, 0.5, '無光學影像', transform=axes[1,0].transAxes, ha='center', fontsize=14)
    axes[1,1].imshow(sar_water, cmap='Blues')
    axes[1,1].set_title('(d) SAR 是唯一來源')

for ax in axes.flat: ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.suptitle('Optical vs SAR — 為什麼需要融合？', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/W10_optical_vs_sar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [S10] Sensor Fusion — Build 4-Class Confidence Map
# ──────────────────────────────────────────────────────────────────
# TODO: 填入正確的 class code（3, 2, 1）
# ──────────────────────────────────────────────────────────────────

fusion = np.zeros((H, W), dtype=np.uint8)
fusion[(ndwi_mask == 1) & (sar_water == 1)] = ___           # <-- TODO: High Confidence
fusion[(cloud_mask == 1) & (sar_water == 1) & (fusion != 3)] = ___  # <-- TODO: SAR Only
fusion[(ndwi_mask == 1) & (sar_water == 0) & (cloud_mask == 0)] = ___  # <-- TODO: Optical Only

labels = {0: 'No Detection', 1: 'Optical Only', 2: 'SAR Only (Cloudy)', 3: 'High Confidence'}

print('Fusion result:')
for v, lbl in labels.items():
    n = np.sum(fusion == v)
    print(f'  {lbl}: {n:,} px = {n * px_area_ha:.1f} ha')

### Topographic Audit: Removing False Positives / 地形審計

SAR 側視幾何在陡峭地形造成假水體：

- **Foreshortening (前縮效應)**: 面向雷達的坡面被壓縮 → 異常低回波
- **Layover (疊置效應)**: 極陡坡面折疊 → 假暗區
- **Radar Shadow (雷達陰影)**: 背向雷達的坡面無信號 → 像水

**規則**: slope > 25-30° → 不可能是積水 → 排除

In [ ]:
# [S11] DEM 地形展示 — 理解堰塞湖地形脈絡
# ── 載入 Copernicus DEM，展示災前地形
#
# ⚠ 重要提醒：
#   Copernicus DEM (2011-2014) 是「災前」地形！
#   崩塌改變了地形 → 舊 DEM 的坡度不再正確
#   因此本案例「不進行」地形校正（沒有災後 DEM）
#
#   實務上，若有災後 DEM（光達或 InSAR 重建），
#   可用 slope > 25-30° 過濾陡坡上的 radar shadow 假水體。

def load_dem(bbox=LAKE_BBOX):
    """Load Copernicus DEM GLO-30 via STAC."""
    search = catalog.search(collections=['cop-dem-glo-30'], bbox=bbox)
    items = list(search.items())
    print(f'  DEM tiles: {len(items)}')
    if not items:
        return np.zeros((H, W), dtype=np.float32)
    signed = [pc.sign(item) for item in items]
    dem_lazy = stackstac.stack(
        signed, assets=['data'], epsg=32651, resolution=10,
        bounds_latlon=bbox, chunksize=2048,
    )
    if dem_lazy.sizes.get('time', 1) > 1:
        dem_arr = safe_compute(dem_lazy.max(dim='time'))
    else:
        dem_arr = safe_compute(dem_lazy.squeeze('time'))
    dem = dem_arr.values.squeeze().astype(np.float32)
    dem[dem <= 0] = np.nan
    if np.any(np.isnan(dem)):
        dem[np.isnan(dem)] = np.nanmedian(dem)
    return dem

def calc_slope(dem, cell_size=10):
    from scipy.ndimage import uniform_filter
    dem_smooth = uniform_filter(dem, size=3)
    dy, dx = np.gradient(dem_smooth, cell_size)
    return np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

print('Loading Copernicus DEM GLO-30 (pre-disaster)...')
dem = load_dem()
slope = calc_slope(dem)

from scipy.ndimage import zoom as scipy_zoom
if slope.shape != (H, W):
    slope = scipy_zoom(slope, (H/slope.shape[0], W/slope.shape[1]), order=1)
    dem = scipy_zoom(dem, (H/dem.shape[0], W/dem.shape[1]), order=1)

print(f'  DEM: {np.nanmin(dem):.0f}–{np.nanmax(dem):.0f} m')
print(f'  Slope: {np.nanmin(slope):.1f}°–{np.nanmax(slope):.1f}°')
print(f'  Steep (> 30°): {np.mean(slope > 30)*100:.0f}%')
print()
print('⚠ 此 DEM 為災前地形，不適用於災後地形校正')
print('  若有災後 DEM，可過濾 slope > 25-30° 的 radar shadow 假水體')
print()
print('📋 討論題：')
print('  Q1: 為什麼不能直接用 Copernicus DEM 過濾堰塞湖的假水體？')
print('  Q2: 災後要用什麼方法取得更新的 DEM？')
print('  Q3: 從災害發生到取得新 DEM，通常需要多久？')

In [ ]:
# [S12] Confidence Map + DEM Context

cmap_f = mcolors.ListedColormap(['#E8E8E8', '#3182CE', '#FF6B2A', '#D24817'])
norm_f = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap_f.N)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# (a) DEM hillshade — 災前地形脈絡
from matplotlib.colors import LightSource
ls = LightSource(azdeg=315, altdeg=45)
hs = ls.hillshade(dem, vert_exag=2, dx=10, dy=10)
axes[0].imshow(hs, cmap='gray')
axes[0].imshow(sar_water, cmap='Blues', alpha=0.5)
axes[0].set_title(f'(a) 災前地形 + SAR 水體\nCopernicus DEM Hillshade', fontsize=12)

# (b) Slope
im_slope = axes[1].imshow(slope, cmap='YlOrRd', vmin=0, vmax=50)
axes[1].set_title(f'(b) 坡度圖（災前 DEM）\n山區坡度 > 30° 區域多', fontsize=12)
plt.colorbar(im_slope, ax=axes[1], shrink=0.8, label='Slope (°)')

# (c) Final confidence map
im = axes[2].imshow(fusion, cmap=cmap_f, norm=norm_f)
axes[2].set_title(f'(c) ARIA v7.0 Confidence Map\n融合信心圖（未地形校正）', fontsize=12)
cbar = plt.colorbar(im, ax=axes[2], ticks=[0,1,2,3], shrink=0.8)
cbar.ax.set_yticklabels(['No Detection', 'Optical Only', 'SAR Only', 'High Conf.'],
                         fontsize=9)

for ax in axes:
    ax.set_xlabel('Column'); ax.set_ylabel('Row')

plt.suptitle(f'ARIA v7.0: 堰塞湖偵測成果',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/W10_L2_confidence_map.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Lab 2 figure saved')

In [ ]:
# [S13] ARIA v7.0 Summary Report
# ──────────────────────────────────────────────────────────────────

labels = {0: 'No Detection 無偵測', 1: 'Optical Only 僅光學',
          2: 'SAR Only (Cloudy) 僅雷達', 3: 'High Confidence 高信心'}

print('=' * 60)
print('ARIA v7.0 ALL-WEATHER INTELLIGENCE REPORT')
print('=' * 60)
print(f'  Case:       馬太鞍溪堰塞湖')
print(f'  SAR date:   {post_item.properties["datetime"][:10]}')
print(f'  Sensor:     Sentinel-1 ({post_item.properties.get("sat:orbit_state","?")})')
print(f'  Threshold:  VV < {SAR_THRESHOLD} dB + morphological cleanup')
print(f'  BBOX:       {LAKE_BBOX}')
print(f'  Source:     Planetary Computer STAC (cloud-streamed)')
print('-' * 60)
for v, lbl in labels.items():
    n = np.sum(fusion == v)
    print(f'  {lbl}: {n:,} px = {n * px_area_ha:.1f} ha')
total_water = np.sum(fusion >= 1)
print(f'  ────────────────────────────────')
print(f'  Total detected: {total_water:,} px = {total_water * px_area_ha:.1f} ha')
print('=' * 60)
print()
print('📌 方法論備註：')
print('  • SAR 門檻 -14 dB（寬鬆）+ morphological opening + connected component ≥ 0.5 ha')
print('  • 未進行地形校正（Copernicus DEM 為災前地形，不適用）')
print('  • 若有災後 DEM，可額外過濾 slope > 30° 的 radar shadow')

---
## 課堂即時挑戰：SAR Threshold Challenge（15–20 分鐘）

> **⏰ 限時練習 — 請在課堂結束前上傳至 NTUCool！**

### 任務

老師將公布一個**第二個 SAR 閾值**（-18 dB，ARIA 文獻預設值）。  
比較兩個不同閾值的淹水偵測結果（都不加形態學清理）。

### 步驟

1. 套用兩個閾值（-14 dB + -18 dB）— 直接 threshold，不加 morphological cleaning
2. 製作 1×2 比較圖
3. 填寫比較表（水體像素數、面積、平均 dB）
4. 寫出判斷：寬鬆門檻 + 後處理 vs 嚴格門檻，哪個策略更適合防災早期預警？為什麼？（2-3 句話）

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 課堂即時挑戰 — SAR Threshold Comparison
# ═══════════════════════════════════════════════════════════════

THRESHOLD_NEW = -18  # <-- ARIA 文獻預設值

mask_default = (sar_filtered < SAR_THRESHOLD).astype(np.uint8)  # -14 dB（寬鬆）
mask_new = (sar_filtered < THRESHOLD_NEW).astype(np.uint8)      # -18 dB（嚴格）

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(mask_default, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title(f'Threshold = {SAR_THRESHOLD} dB（寬鬆）\n{np.sum(mask_default):,} px = {np.sum(mask_default)*px_area_ha:.1f} ha')
axes[1].imshow(mask_new, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title(f'Threshold = {THRESHOLD_NEW} dB（嚴格）\n{np.sum(mask_new):,} px = {np.sum(mask_new)*px_area_ha:.1f} ha')
for ax in axes: ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.suptitle('SAR Threshold Comparison — 不加形態學清理', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 填寫比較表 ──
print(f'Threshold {SAR_THRESHOLD} dB: {np.sum(mask_default):,} px, {np.sum(mask_default)*px_area_ha:.1f} ha, '
      f'mean dB = {np.nanmean(sar_filtered[mask_default==1]):.1f}')
print(f'Threshold {THRESHOLD_NEW} dB: {np.sum(mask_new):,} px, {np.sum(mask_new)*px_area_ha:.1f} ha, '
      f'mean dB = {np.nanmean(sar_filtered[mask_new==1]):.1f}')
print(f'\n差異: {np.sum(mask_default)-np.sum(mask_new):,} px '
      f'({(np.sum(mask_default)-np.sum(mask_new))*px_area_ha:.1f} ha)')
print(f'NCDR 實測面積約 80 ha')
print()
print('# ── 你的判斷（2-3 句話）──')
print('# TODO: 哪個策略更適合防災早期預警？')
print('# 寬鬆門檻(-14) + morphological 清理 vs 嚴格門檻(-18)')

---
## Wrap-Up Checklist / 實驗完成清單

### Lab 1
- [ ] STAC 搜尋 `sentinel-1-rtc`（災前 + 災後）
- [ ] `stream_sar()` 串流載入 VV
- [ ] Linear → dB 轉換（`10 * np.log10()`）
- [ ] Speckle filter（Median 5×5）
- [ ] Water mask（VV < SAR_THRESHOLD）
- [ ] Before/After 對比圖

### Lab 2
- [ ] Sentinel-2 NDWI + Cloud mask
- [ ] 4-class fusion map
- [ ] Topographic correction（slope > 25-30°）
- [ ] ARIA v7.0 Report

### W8–W10 完整鏈結
- 同一個 STAC → stackstac → xarray 工作流
- W8–W9: Sentinel-2（光學）
- W10: Sentinel-1（SAR）+ 兩者融合
- **不需要下載任何檔案 — 全部雲端串流！**

---
### 驗證你的成果

| Check | Question |
|-------|----------|
| Flood area | 結果合理嗎？如果 90% 都是水，一定有問題 |
| Speckle | 有先濾波再做閾值嗎？ |
| Topo filter | 山頂上有「水體」嗎？→ 需要地形校正 |
| Fusion | 能解釋為什麼某像素是 High Confidence vs SAR Only 嗎？ |

**跑出來 ≠ 跑對了。不懂可以問，但不要敷衍交差。**